<a href="https://colab.research.google.com/github/jfcrown7-dev/brain-test/blob/main/connect-ai/surgery-slerp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔀 두 AI 섞기 (Blend) — 무료 Colab

비개발자도 **런타임 → 모두 실행**만 누르면 돼요. 결과가 내 HuggingFace에 올라가고, Connect AI 앱에서 "받기"로 바로 씁니다.

> 💎 멤버십은 앱에서 버튼 하나(우리 GPU). 여기선 **무료 Colab GPU**로 직접 — 누구나 AI를 *소유*.


## 🔀 두 AI를 섞어 새 AI 만들기
`결과 = 원본 + 강도×(상대 − 원본)` — 두 모델 가중치를 섞습니다.


In [ ]:
%%capture
!pip install -q torch transformers huggingface_hub peft accelerate safetensors


## 🔑 HuggingFace 로그인 (무료)
결과를 저장할 **무료 HuggingFace 계정**이 필요해요(결제 X). 아래에서 1분이면 됩니다:
1. [huggingface.co 무료 가입](https://huggingface.co/join) → 2. [**write** 토큰 만들기](https://huggingface.co/settings/tokens) → 3. 아래 칸에 붙여넣기


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


## ⚙️ 설정 — 이 4개만 보면 돼요
`MODEL_A`=원본 · `MODEL_B`=상대(능력) · `METHOD` · `SCALE`(강도)


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import HfApi

MODEL_A = "Qwen/Qwen2.5-Coder-0.5B-Instruct"   # 원본
MODEL_B = "Qwen/Qwen2.5-0.5B"   # 상대(능력) 모델
METHOD  = "slerp"   # task_add | task_sub | blend
SCALE   = 0.66       # 강도(λ). 1.0 권장
OUTPUT  = "jfcrown7/SLERP"  # 결과가 올라갈 내 HF repo

def load(repo):
    files = HfApi().list_repo_files(repo)
    full = any(f.endswith(".safetensors") and not f.startswith("adapter") for f in files)
    if (not full) and ("adapter_config.json" in files):
        from peft import AutoPeftModelForCausalLM
        print("🔧 LoRA 어댑터 → 원본에 자동 병합:", repo)
        return AutoPeftModelForCausalLM.from_pretrained(repo, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True).merge_and_unload()
    return AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)

# 방식에 따라 기준(base)·상대(other) 정하기 — 빼기는 능력모델(B)이 기준
base_id, other_id = (MODEL_B, MODEL_A) if METHOD == "task_sub" else (MODEL_A, MODEL_B)
print("📥 두 모델 로딩…")
base = load(base_id); other = load(other_id)

# 핵심: 결과 = base + SCALE×(other − base)  (Task Arithmetic / 블렌드 공통, 구조 무관)
o = dict(other.named_parameters()); applied = 0
with torch.no_grad():
    for n, p in base.named_parameters():
        q = o.get(n)
        if q is not None and q.shape == p.shape:
            p.add_((q.data.to(p.dtype) - p.data) * float(SCALE)); applied += 1
print(f"✅ {applied}개 가중치에 적용 (방식={METHOD}, 강도={SCALE})")

# 결과를 내 HuggingFace에 업로드
base.push_to_hub(OUTPUT)
AutoTokenizer.from_pretrained(base_id).push_to_hub(OUTPUT)
print(f"🎉 완료 → https://huggingface.co/{OUTPUT}")
print("👉 Connect AI 앱 → 🤖 내 AI 에서 받아서 바로 쓰세요!")


📥 두 모델 로딩…


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

✅ 290개 가중치에 적용 (방식=slerp, 강도=0.66)


HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a3c8b49-5a1937583df4db247b30361a;2e70a20a-5fe8-4da7-b216-e297f568beaf)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

## 🎉 끝!
결과가 **내 HuggingFace**에 올라갔어요. Connect AI 앱에서 받아 쓰면 됩니다.
더 쉽게 하려면 → 💎 멤버십(앱에서 버튼 하나).
